# 7. Fine-tuning a pretrained backbone

The training loop here is written out in full rather than hidden behind a trainer. That
is deliberate: fine-tuning is the part people need to adapt, and a loop you can read is
easier to change than a function you have to configure.

The task is binary classification: for each 600 nt tile and each RBP, is the tile bound?
This is a different head from the signal model's, which is the point. The backbone is
reusable; the head is what you replace.

**Report AUROC and AUPRC, never accuracy.** The positive rate here is about 0.5 per cent,
so a model that predicts "unbound" for everything scores 99.5 per cent accuracy and is
worthless. This notebook shows that number explicitly so the trap is visible once.

In [ ]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## The data

`seq` is uint8 with A, C, G, T, N encoded as 0 to 4. `labels` is a boolean matrix of
tiles by RBPs. The file carries its own ordered `rbps` list, so no separate task config is
needed here.

In [ ]:
train = torch.load(pt.classification_path('hepg2_train'), map_location='cpu', weights_only=False)
valid = torch.load(pt.classification_path('hepg2_valid'), map_location='cpu', weights_only=False)

print('keys     :', sorted(train))
print('train seq:', tuple(train['seq'].shape), train['seq'].dtype)
print('labels   :', tuple(train['labels'].shape), train['labels'].dtype)
print('RBPs     :', len(train['rbps']), train['rbps'][:4], '...')

positive_rate = train['labels'].float().mean().item()
print(f'\npositive rate: {positive_rate:.4f}')
print(f'always-negative accuracy would be {1 - positive_rate:.2%}  <- why accuracy is useless here')

### Pick one RBP

One task keeps the notebook fast and the metrics readable. Everything below generalises
to all 103 by widening the head.

In [ ]:
# Pick the RBP with the most positives. With a 0.3 per cent positive rate, a sparse
# task leaves only a few dozen positives in any subset and the metrics become noise.
counts = train['labels'].sum(dim=0)
task = int(counts.argmax())
rbp = train['rbps'][task]
print(f'{rbp}: train positives {int(counts[task])} of {len(train["labels"])} '
      f'({counts[task] / len(train["labels"]):.3%})  <- densest of {len(train["rbps"])} RBPs')


def onehot(seq_uint8):
    """(N, L) uint8 with N=4 -> (N, 4, L) float. The N class becomes an all-zero column,
    which contributes nothing to any convolution, so padding needs no mask."""
    onehot = torch.zeros(seq_uint8.shape[0], 5, seq_uint8.shape[1])
    onehot.scatter_(1, seq_uint8.long().unsqueeze(1), 1.0)
    return onehot[:, :4]


# A subset keeps this notebook to a couple of minutes. Raise these for a real run.
n_train, n_valid = 40000, 10000
rng = np.random.default_rng(0)
train_idx = rng.choice(len(train['seq']), n_train, replace=False)
valid_idx = rng.choice(len(valid['seq']), n_valid, replace=False)

X_train = onehot(train['seq'][train_idx])
y_train = train['labels'][train_idx, task].float()
X_valid = onehot(valid['seq'][valid_idx])
y_valid = valid['labels'][valid_idx, task].float()
print('X_train', tuple(X_train.shape), '| positives', int(y_train.sum()))

## Backbone plus a new head

Load the pretrained model, keep `stem` and `body`, and attach a classification head. The
signal head is discarded: it predicts a 51-task profile and we want one logit per tile.

Note what the head reads. `embed()` returns the representation the signal head would have
received, so its width is the body width here, and it is a `(B, C, L)` feature map. The
head pools it to one vector per tile.

In [ ]:
from parnet.models import RBPNet
from parnet.finetuning import stage_names, set_trainable, describe_trainable
import torch.nn as nn

backbone = RBPNet.from_pretrained(pt.model_path('parnet-3m-pretrain-E2'))
embedding_dim = backbone.embed(torch.zeros(1, 4, 600)).shape[1]
print('embedding width the head will see:', embedding_dim)


class BindingClassifier(nn.Module):
    """Pretrained backbone, mean-pooled, with a linear head. One logit per tile."""

    def __init__(self, backbone, embedding_dim, n_tasks=1):
        super().__init__()
        # Drop the pretrained SIGNAL head. embed() never calls it, so leaving it attached
        # costs nothing at runtime but keeps 80,307 parameters in model.parameters(): they
        # would be handed to the optimizer and counted as trainable while receiving no
        # gradient. Removing it makes "head only" mean what it says.
        backbone.head = None
        self.backbone = backbone
        self.head = nn.Linear(embedding_dim, n_tasks)

    def forward(self, x):
        features = self.backbone.embed(x)      # (B, C, L)
        pooled = features.mean(dim=-1)         # (B, C)
        return self.head(pooled).squeeze(-1)   # (B,)


model = BindingClassifier(backbone, embedding_dim).to(device)
print(model.head)

## Choosing what to train

"Unfreeze the last N layers" breaks as soon as a model may or may not have a trainable
projection: the last trainable thing is sometimes `body[-1]` and sometimes `projection`.
`parnet.finetuning` counts **stages that own parameters** instead, so a parameterless
projection never consumes a slot and the same call means the same thing on every model.

In [ ]:
print('stages:', stage_names(backbone))

# Head only: the cheapest option and the right default on a small dataset.
set_trainable(backbone, unfreeze_last=0)
counts = describe_trainable(model)
print('\nhead only          ->', counts)

# Last two stages as well, for comparison.
set_trainable(backbone, unfreeze_last=2)
print('plus last 2 stages ->', describe_trainable(model))

# Back to head only for the run below.
set_trainable(backbone, unfreeze_last=0)
trainable = [p for p in model.parameters() if p.requires_grad]
print('\ntraining', sum(p.numel() for p in trainable), 'parameters')

## The loop

Plain PyTorch, written out. `pos_weight` compensates the class imbalance so the gradient
is not dominated by negatives.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

batch_size = 128
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

pos_weight = ((y_train == 0).sum() / (y_train == 1).sum()).clamp(max=100.0)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.AdamW(trainable, lr=1e-3)
print(f'pos_weight {pos_weight:.1f}')

model.train()
for epoch in range(2):
    running, seen = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(yb)
        seen += len(yb)
    print(f'epoch {epoch}: train loss {running / seen:.4f}')

## Evaluate: AUROC and AUPRC, with the baseline alongside

AUPRC's baseline is the positive rate, so an AUPRC of 0.05 can be excellent or terrible
depending on prevalence. Always print the prevalence next to it.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

model.eval()
scores = []
with torch.no_grad():
    for start in range(0, len(X_valid), 256):
        batch = X_valid[start:start + 256].to(device)
        scores.append(model(batch).cpu())
scores = torch.cat(scores).numpy()
truth = y_valid.numpy()

prevalence = truth.mean()
auroc = roc_auc_score(truth, scores)
auprc = average_precision_score(truth, scores)

print(f'{rbp}')
print(f'  prevalence (AUPRC baseline) {prevalence:.4f}')
print(f'  AUROC                       {auroc:.4f}   (0.5 is chance)')
print(f'  AUPRC                       {auprc:.4f}   (lift {auprc / prevalence:.1f}x over baseline)')
print()
print(f'  accuracy of "always unbound" {1 - prevalence:.2%}  <- reports success, learned nothing')

## What to take away

1. The backbone is reusable; the head is what you replace.
2. Ask for the embedding width rather than hard-coding it: it differs between model sizes
   and changes again if a projection is ever trained.
3. Unfreeze by stages that own parameters, not by counting layers.
4. Report AUROC and AUPRC with the prevalence beside them. Accuracy at a 0.5 per cent
   positive rate is a number that cannot fail.